# Практическая работа 6. Задание 3
Арбитраж валют с помощью Беллмана-Форда

In [1]:
import math

# курсы обмена: (из, в, курс)
rates = [
    ('USD', 'EUR', 0.85),
    ('EUR', 'GBP', 1.15),
    ('GBP', 'USD', 1.05),
    ('USD', 'JPY', 145),
    ('JPY', 'GBP', 0.0075)
]
currencies = ['USD', 'EUR', 'GBP', 'JPY']
idx = {c: i for i, c in enumerate(currencies)}
# вес ребра = -log(курс)
edges = [(idx[u], idx[v], -math.log(r)) for u, v, r in rates]
n = len(currencies)


def find_negative_cycle(edges, n, start):
    INF = float('inf')
    dist = [INF] * n
    prev = [None] * n
    dist[start] = 0
    for _ in range(n - 1):
        for u, v, w in edges:
            if dist[u] != INF and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
    # проверочный проход: если ребро еще релаксируется, цикл есть
    x = None
    for u, v, w in edges:
        if dist[u] != INF and dist[u] + w < dist[v]:
            prev[v] = u
            x = v
            break
    if x is None:
        return None
    # идем назад по prev, пока вершина не повторится
    path = []
    while x not in path:
        path.append(x)
        x = prev[x]
    cycle = path[path.index(x):] + [x]
    return cycle[::-1]


def product(names):
    # произведение курсов по циклу
    p = 1
    for i in range(len(names) - 1):
        for u, v, r in rates:
            if u == names[i] and v == names[i + 1]:
                p *= r
    return p

In [2]:
cycle = find_negative_cycle(edges, n, idx['USD'])
if cycle is None:
    print('Арбитража нет')
else:
    names = [currencies[v] for v in cycle]
    # начнем цикл с USD
    if 'USD' in names:
        k = names.index('USD')
        names = names[k:-1] + names[:k] + ['USD']
    print('Найден отрицательный цикл:', ' -> '.join(names))
    p = product(names)
    print('Произведение курсов:', round(p, 6))
    print('Стартовый капитал: 1000')
    print('После обмена по циклу:', round(1000 * p, 2))
    print('Прибыль:', round(1000 * p - 1000, 2))

print()
# на рисунке два цикла через USD, проверим оба
for c in [['USD', 'EUR', 'GBP', 'USD'], ['USD', 'JPY', 'GBP', 'USD']]:
    p = product(c)
    print(' -> '.join(c), ': произведение', round(p, 6))
    print('    из 1000 будет', round(1000 * p, 2))

Найден отрицательный цикл: USD -> JPY -> GBP -> USD
Произведение курсов: 1.141875
Стартовый капитал: 1000
После обмена по циклу: 1141.88
Прибыль: 141.88

USD -> EUR -> GBP -> USD : произведение 1.026375
    из 1000 будет 1026.38
USD -> JPY -> GBP -> USD : произведение 1.141875
    из 1000 будет 1141.88
